# Auto Insurance Fraud Detection with Machine Learning

This analysis compares four classification approaches for auto-insurance fraud detection:

- Logistic Regression
- Support Vector Machine (SVM)
- Random Forest
- XGBoost

A common preprocessing and SMOTE pipeline is used for each model. Hyperparameters are selected using 5-fold stratified cross-validation with ROC AUC as the selection metric. Final performance is assessed on a held-out test set.


In [ ]:
# Google Colab setup
from pathlib import Path
REPO_URL = "https://github.com/seanhutagaol/Auto-Insurance-Fraud-ML.git"
REPO_DIR = Path("/content/Auto-Insurance-Fraud-ML")

if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR

%cd /content/Auto-Insurance-Fraud-ML
!pip install -q imbalanced-learn xgboost

import sys
sys.path.append(str(REPO_DIR))


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

from src.data_processing import (
    load_and_clean_data, remove_outliers_zscore,
    feature_engineering, get_train_test_split
)
from src.pipelines import build_model_pipeline
from src.evaluation import (
    evaluate_model, plot_confusion_matrix,
    plot_roc_curves, results_table
)


## 1. Data loading and preprocessing

In [ ]:
filepath = "data/insurance.csv"

df = load_and_clean_data(filepath)
print("Raw shape:", df.shape)
display(df.head())


In [ ]:
df = remove_outliers_zscore(df, threshold=3.0)
df = feature_engineering(df)

print("Processed shape:", df.shape)
print(f"Fraud rate: {df['fraud_reported'].mean():.2%}")
display(df.head())


In [ ]:
X_train, X_test, y_train, y_test = get_train_test_split(df)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print(f"Training fraud rate: {y_train.mean():.2%}")
print(f"Testing fraud rate: {y_test.mean():.2%}")


## 2. Four model pipelines

In [ ]:
models = {
    "Logistic Regression": build_model_pipeline(
        LogisticRegression(max_iter=3000, random_state=42)
    ),
    "SVM": build_model_pipeline(
        SVC(probability=True, random_state=42)
    ),
    "Random Forest": build_model_pipeline(
        RandomForestClassifier(random_state=42, n_jobs=-1)
    ),
    "XGBoost": build_model_pipeline(
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    ),
}

list(models.keys())


## 3. Hyperparameter tuning with 5-fold stratified cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__solver": ["liblinear", "lbfgs"],
        "classifier__penalty": ["l2"],
    },
    "SVM": {
        "classifier__C": [0.1, 1, 10],
        "classifier__kernel": ["rbf", "linear"],
        "classifier__gamma": ["scale", "auto"],
    },
    "Random Forest": {
        "classifier__n_estimators": [200, 400],
        "classifier__max_depth": [None, 10, 20],
        "classifier__min_samples_split": [2, 5],
        "classifier__min_samples_leaf": [1, 2],
    },
    "XGBoost": {
        "classifier__n_estimators": [200, 400],
        "classifier__max_depth": [3, 5],
        "classifier__learning_rate": [0.03, 0.1],
        "classifier__subsample": [0.8, 1.0],
        "classifier__colsample_bytree": [0.8, 1.0],
    },
}


In [ ]:
best_models = {}
cv_records = []

for name, pipeline in models.items():
    print(f"\n{'='*70}\nTuning {name}\n{'='*70}")

    search = GridSearchCV(
        pipeline,
        param_grid=param_grids[name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1
    )
    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_
    cv_records.append({
        "Model": name,
        "Best CV ROC AUC": search.best_score_,
        "Best Parameters": search.best_params_
    })

    print("Best CV ROC AUC:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

cv_summary = pd.DataFrame(cv_records)
display(cv_summary)


## 4. Held-out test-set evaluation

In [ ]:
results = [
    evaluate_model(model, X_test, y_test, model_name=name)
    for name, model in best_models.items()
]

comparison = results_table(results)
display(comparison)


In [ ]:
for name, model in best_models.items():
    plot_confusion_matrix(model, X_test, y_test, model_name=name)


In [ ]:
plot_roc_curves(best_models, X_test, y_test)


## 5. Interpretation

The four models represent complementary modelling approaches. Logistic Regression provides a linear and relatively interpretable baseline; SVM provides a kernel-based nonlinear classifier; Random Forest captures nonlinear interactions through an ensemble of trees; and XGBoost uses gradient-boosted decision trees.

Among the four models, **XGBoost achieved the strongest overall predictive performance**, with a ROC AUC of 0.804, accuracy of 0.798, sensitivity of 0.796, specificity of 0.799, and F1-score of 0.661. Its relatively balanced sensitivity and specificity indicate that it can identify fraudulent claims while maintaining reasonable discrimination against legitimate claims.

Random Forest achieved a higher specificity (0.926) but substantially lower sensitivity (0.245), indicating that it was considerably more conservative in identifying fraudulent claims and consequently missed a larger proportion of fraudulent cases. Logistic Regression and SVM produced lower discrimination, with ROC AUC values of approximately 0.62.

For fraud detection, sensitivity is particularly important because false negatives correspond to fraudulent claims that are missed, while specificity reflects the ability to avoid incorrectly flagging legitimate claims. ROC AUC provides a threshold-independent summary of ranking performance.

Overall, the results suggest that **XGBoost provides the most effective balance between fraud detection and false-positive control among the evaluated models**.

All preprocessing and SMOTE operations are contained within the model pipelines, so they are fitted as part of the cross-validation workflow rather than being applied globally before model selection.


## 6. Conclusion

Among the four evaluated models, **XGBoost provided the strongest overall performance**, achieving the highest ROC AUC (0.804), accuracy (0.798), sensitivity (0.796), and F1-score (0.661), while maintaining a specificity of 0.799. These results indicate that XGBoost provides the most effective balance between identifying fraudulent claims and limiting false positives in this dataset.

Random Forest demonstrated high specificity (0.926) but substantially lower sensitivity (0.245), making it less effective for detecting fraudulent claims despite its ability to avoid false-positive classifications. Logistic Regression and SVM showed comparatively weaker discrimination, with ROC AUC values of approximately 0.62.

Overall, the results support **XGBoost as the preferred model among the four evaluated approaches** for this auto-insurance fraud detection task. Future work could investigate probability-threshold optimization and additional feature engineering to further align model performance with the operational costs of false-positive and false-negative fraud decisions.
